# 1. What are we actually trying to do?

We have prices:

```text
[7, 1, 5, 3, 6, 4]
```

Each number is the stock price on a different day.

We can:

* Buy **once**
* Sell **once**
* We must buy **before** selling

We want the **maximum profit**.

For example:

```text
Buy at 1
Sell at 6

Profit = 6 - 1 = 5
```

So answer:

```text
5
```

---

# 2. First: why brute force?

The most obvious thought is:

> "I'll try every possible day to buy and every possible day after that to sell."

For:

```text
[7, 1, 5, 3, 6, 4]
```

we try:

```text
Buy 7 → sell 1
Buy 7 → sell 5
Buy 7 → sell 3
Buy 7 → sell 6
Buy 7 → sell 4

Buy 1 → sell 5
Buy 1 → sell 3
Buy 1 → sell 6
Buy 1 → sell 4

...
```

For every pair:

```text
profit = selling_price - buying_price
```

Keep the biggest one.

This works.

But there's a problem.

---

# 3. WHY is brute force slow?

Imagine we have:

```text
1000 days
```

For day 1, we check almost 1000 future days.

For day 2, we check almost 999.

For day 3, almost 998...

We're repeatedly asking:

> "What is the best buying price before this selling day?"

We are doing **far more checking than necessary**.

That's why brute force becomes:

```text
O(n²)
```

---

# 4. Now let's think differently

Instead of asking:

> "For every buying day, what are all the possible selling days?"

Let's reverse the thinking.

Ask:

> **"If today is my selling day, what is the best day I could have bought before today?"**

This is much easier.

Suppose today the price is:

```text
6
```

What do I need to know?

I don't care about every previous price individually.

I only care about:

> **What was the cheapest price I could have bought at before today?**

Because buying cheaper = making more profit.

Suppose previous prices were:

```text
7, 1, 5, 3
```

The cheapest is:

```text
1
```

So if today's price is `6`:

```text
profit = 6 - 1
       = 5
```

That's all we need.

---

# 5. This gives us the key idea

As we move through the array, maintain:

```text
minimum_price
```

meaning:

> **The cheapest stock price I've seen so far.**

And whenever we see today's price, calculate:

```text
today's profit = today's price - minimum_price
```

Then keep the biggest profit we've seen.

So we only need **two things**:

```text
minimum_price → cheapest buying price so far

max_profit → best profit found so far
```

---

# 6. Why does this actually work?

This is the most important part.

Let's say we're currently at:

```text
6
```

Previous prices:

```text
7, 1, 5, 3
```

Possible buying prices:

```text
7
1
5
3
```

Which one gives us the biggest profit when selling at `6`?

Obviously:

```text
1
```

because:

```text
6 - 1 = 5
6 - 3 = 3
6 - 5 = 1
6 - 7 = -1
```

So instead of trying all four:

```text
7
1
5
3
```

we just remember:

```text
minimum_price = 1
```

Then:

```text
6 - 1 = 5
```

**That's the optimization.**

We're throwing away information we don't need.

---

# 7. Why can we safely throw away the other prices?

This is the key reasoning.

Suppose we've already seen:

```text
1
```

Then later we see:

```text
3
```

Could `3` ever be a better buying price than `1`?

No.

If we eventually sell at `10`:

```text
10 - 1 = 9
10 - 3 = 7
```

Buying at `1` is always better.

Therefore, once we know:

```text
minimum_price = 1
```

we don't need to remember `3`, `5`, or `7` as possible better buying prices.

**Only the minimum matters.**

That's why the algorithm can work in one pass.

---

# 8. Let's walk through the entire array

```text
[7, 1, 5, 3, 6, 4]
```

Initially:

```text
minimum_price = 7
max_profit = 0
```

We haven't seen anything else.

---

## Day 2 → price = 1

Ask:

> Is `1` cheaper than my current minimum `7`?

Yes.

So:

```text
minimum_price = 1
```

We don't make a profit here because we can't buy and sell meaningfully at the same point.

Now:

```text
minimum_price = 1
max_profit = 0
```

---

## Day 3 → price = 5

Now ask:

> If I bought at the cheapest price I've seen (`1`) and sold today at `5`, what do I make?

```text
5 - 1 = 4
```

So:

```text
max_profit = 4
```

Is `5` cheaper than our minimum `1`?

No.

So:

```text
minimum_price = 1
```

---

## Day 4 → price = 3

Profit:

```text
3 - 1 = 2
```

We already have:

```text
max_profit = 4
```

So keep `4`.

Minimum is still `1`.

---

## Day 5 → price = 6

Profit:

```text
6 - 1 = 5
```

This is better than our previous `4`.

So:

```text
max_profit = 5
```

---

## Day 6 → price = 4

Profit:

```text
4 - 1 = 3
```

Not better than `5`.

So final:

```text
max_profit = 5
```

---

# 9. Now the code makes sense

```python
def max_profit(prices):
    minimum_price = prices[0]
    max_profit = 0

    for i in range(1, len(prices)):
        profit = prices[i] - minimum_price

        max_profit = max(max_profit, profit)

        minimum_price = min(minimum_price, prices[i])

    return max_profit
```

The code is basically implementing the exact reasoning we just did.

---

## Line 1

```python
minimum_price = prices[0]
```

We need some initial buying price.

So we start with the first day's price.

---

## Line 2

```python
max_profit = 0
```

Initially, no transaction has been made.

Also, if prices only decrease, the correct answer is `0`.

Example:

```text
[7, 6, 5, 4]
```

There is no profitable transaction.

---

## Line 3

```python
for i in range(1, len(prices)):
```

Start from the second day.

Why?

Because the first day is already stored in:

```python
minimum_price
```

---

## Line 4

```python
profit = prices[i] - minimum_price
```

This is asking:

> "If I sell **today**, and buy at the cheapest price I've seen before today, what profit do I get?"

That's the entire purpose of this line.

---

## Line 5

```python
max_profit = max(max_profit, profit)
```

Maybe today's profit is the best we've seen.

So compare:

```text
old best
vs
today's profit
```

Keep the larger one.

---

## Line 6

```python
minimum_price = min(minimum_price, prices[i])
```

Now ask:

> "Is today's price cheaper than everything I've seen before?"

If yes, today's price becomes our new buying price for **future days**.

---

# 10. WHY do we calculate profit before updating minimum?

This is subtle and important.

Suppose:

```text
prices = [7, 1]
```

When we reach `1`, we should say:

```text
1 is a great buying price for the FUTURE.
```

We cannot say:

```text
Buy at 1
Sell at 1
```

That gives no profit and isn't the intended buy-before-sell transaction.

So we first use the old minimum to calculate today's possible profit:

```python
profit = prices[i] - minimum_price
```

Then we update:

```python
minimum_price = min(minimum_price, prices[i])
```

Meaning:

> "Today's price can now be used as a buying price for future days."

So the order is important.

---

# 11. Why is it O(n), not O(n²)?

This is the beautiful part.

Brute force does:

```text
For every day:
    check many future days
```

So:

```text
O(n²)
```

Our approach does:

```text
Look at each day exactly once.
```

At each day, we do only a few operations:

```text
1. Calculate profit
2. Update max profit
3. Update minimum price
```

So:

```text
n days × constant work
= O(n)
```

And we only use two variables:

```text
minimum_price
max_profit
```

So extra space:

```text
O(1)
```

---

# The intuition you should remember

Don't memorize the code.

Remember this conversation happening inside your head:

> **"Today's price is X. What's the cheapest price I've seen before today?"**

Then:

> **"If I bought at that cheapest price and sell today, what's my profit?"**

Then:

> **"Is this the best profit I've seen?"**

Then:

> **"Is today's price the new cheapest price for future days?"**

That's the whole algorithm.

### In one sentence:

**Keep the cheapest price seen so far, use it to calculate the profit you could make today, and remember the biggest profit.**

